# Classification automatique des rappels de produits (RappelConso)

**Objectif :** à partir du texte d'une fiche de rappel (motif du rappel, nom du produit, catégorie), prédire la ou les **familles de risque** :
microbiologique, chimique, allergène, corps étranger, blessure / physique, incendie / électrique, étiquetage / qualité.

**Données :** RappelConso V2 (DGCCRF, DGAL…), data.gouv.fr, Licence Ouverte 2.0.

**Démarche :**
1. Préparation des données et construction des étiquettes (familles de risque)
2. Découpage **temporel** : entraînement ≤ 2024, validation = 2025, test = 2026
3. Baseline : TF-IDF + régression logistique
4. Modèle principal : **CamemBERT** fine-tuné en **PyTorch** (classification multi-label)
5. Suivi des expériences avec **MLflow**, comparaison et analyse d'erreurs
6. Export du modèle pour l'API

> Avant de lancer : menu **Exécution → Modifier le type d'exécution → GPU (T4)**.

## 0. Installation et vérification du GPU

In [ ]:
!pip -q install transformers datasets accelerate mlflow scikit-learn

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Chargement des données

Télécharge le CSV depuis data.gouv.fr (« RappelConso V2 ») puis charge-le ici avec le bouton qui apparaît.

In [ ]:
from google.colab import files
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print(CSV_PATH)

In [ ]:
import re, json, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8-sig", dtype=str)
df.columns = [c.replace("﻿", "") for c in df.columns]  # nettoie le BOM dans les noms de colonnes
df["date_publication"] = pd.to_datetime(df["date_publication"], utc=True)
df = df.drop_duplicates("numero_fiche", keep="last")
print(df.shape)
df[["date_publication", "categorie_produit", "libelle", "motif_rappel", "risques_encourus"]].head()

## 2. Construction des étiquettes : familles de risque

La colonne `risques_encourus` contient plus de 300 valeurs différentes (et parfois plusieurs risques séparés par `|`).
On les regroupe en **7 familles**. Une fiche peut appartenir à plusieurs familles → problème **multi-label**.

Les fiches dont le seul risque est « autre risque » sont **retirées** : l'étiquette ne dit rien du risque réel, elle ajouterait du bruit.
En production, si le modèle ne dépasse le seuil pour aucune famille, la fiche est renvoyée en « autre / à vérifier par un humain ».

In [ ]:
FAMILLES = {
    "microbiologique": r"listeria|salmonell|escherichia|stec|clostridium|norovirus|staphylo|bacillus|biologique|microbiolog|moisissure|hépatite|vibrio|cronobacter|campylobacter|yersinia|parasit|levure|brucella|pseudomonas",
    "chimique": r"chimique|pesticide|phytosanitaire|substance.? interdite|additif|aflatoxine|histamine|datura|métaux|mycotoxine|ochratoxine|oxyde d.éthylène|alcaloïde|toxine|dioxine|intoxication|hydrocarbures|médicaments vétérinaires|patuline|ogm|dépassement de seuils",
    "allergene": r"allerg",
    "corps_etranger": r"inertes|corps étranger",
    "blessure_physique": r"blessure|étouffement|dommages internes|brûlure|arrêt respiratoire|strangulation|étranglement|etranglement|suffocation|asphyxie|coincement|chute|écrasement|coupure|noyade",
    "incendie_electrique": r"incendie|choc électrique|électrocution|explosion|électromagnétique",
    "etiquetage_qualite": r"étiquetage|défaut de fabrication|conditionnement|organoleptique|non.conformité",
}
LABELS = list(FAMILLES.keys())

def familles(risques):
    r = str(risques).lower()
    return [f for f, motif in FAMILLES.items() if re.search(motif, r)]

df["familles"] = df["risques_encourus"].apply(familles)
print("Fiches sans famille (« autre ») retirées :", (df["familles"].str.len() == 0).sum())
df = df[df["familles"].str.len() > 0].copy()

for lab in LABELS:
    df[lab] = df["familles"].apply(lambda fs: int(lab in fs))

print("Fiches restantes :", len(df))
print("Part multi-label :", round((df[LABELS].sum(axis=1) > 1).mean(), 3))
df[LABELS].sum().sort_values(ascending=False)

## 3. Texte d'entrée

On construit le texte à partir de ce qu'on connaît **au moment où le rappel est rédigé** : le motif, le nom du produit et sa catégorie.

On exclut volontairement `risques_encourus` (c'est l'étiquette), `preconisations_sanitaires` et `description_complementaire_risque` :
ils décrivent directement le risque et créeraient une **fuite de données** (le modèle « tricherait »).

In [ ]:
def nettoyer(t):
    t = str(t) if pd.notna(t) else ""
    t = t.replace("¤", " ")
    return re.sub(r"\s+", " ", t).strip()

df["texte"] = (
    "Produit : " + df["libelle"].map(nettoyer)
    + " | Catégorie : " + df["categorie_produit"].map(nettoyer) + " / " + df["sous_categorie_produit"].map(nettoyer)
    + " | Motif : " + df["motif_rappel"].map(nettoyer)
)
print(df["texte"].str.len().describe())
for t in df["texte"].sample(3, random_state=SEED):
    print("-", t)

## 4. Découpage temporel

Comme en production, on entraîne sur le **passé** et on évalue sur le **futur** :
- entraînement : fiches publiées jusqu'en 2024
- validation (choix du meilleur modèle et du seuil) : 2025
- test (score final, jamais utilisé pour les réglages) : 2026

In [ ]:
annee = df["date_publication"].dt.year
train_df = df[annee <= 2024].reset_index(drop=True)
val_df = df[annee == 2025].reset_index(drop=True)
test_df = df[annee == 2026].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

Y_train, Y_val, Y_test = (d[LABELS].values for d in (train_df, val_df, test_df))
pd.DataFrame({"train": Y_train.mean(0), "val": Y_val.mean(0), "test": Y_test.mean(0)}, index=LABELS).round(3)

## 5. Suivi des expériences avec MLflow

Chaque modèle (baseline, CamemBERT) est enregistré comme un « run » MLflow : paramètres, métriques, artefacts.
Les résultats sont stockés dans une petite base SQLite (`mlflow.db`, plus le dossier `mlruns/` pour les fichiers), qu'on téléchargera à la fin pour les consulter sur ton ordi avec `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("rappelconso-familles-risque")

from sklearn.metrics import f1_score, classification_report

def scores(y_true, y_pred):
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def predire_avec_seuil(probas, seuil):
    return (probas >= seuil).astype(int)

def meilleur_seuil(y_true, probas):
    # seuil global choisi sur la validation (jamais sur le test)
    grille = np.arange(0.20, 0.75, 0.05)
    res = [(s, f1_score(y_true, predire_avec_seuil(probas, s), average="macro", zero_division=0)) for s in grille]
    return max(res, key=lambda x: x[1])[0]

## 6. Baseline : TF-IDF + régression logistique

Un modèle simple et rapide : il compte les mots et groupes de 2 mots (TF-IDF) et apprend une régression logistique par famille (one-vs-rest).
C'est le score que CamemBERT doit battre.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

params_base = {"ngram_range": "(1,2)", "min_df": 2, "C": 5, "class_weight": "balanced"}
vect = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vect.fit_transform(train_df["texte"])
X_val, X_test = vect.transform(val_df["texte"]), vect.transform(test_df["texte"])

baseline = OneVsRestClassifier(LogisticRegression(max_iter=2000, C=5, class_weight="balanced"))
baseline.fit(X_train, Y_train)

proba_val_base = baseline.predict_proba(X_val)
seuil_base = meilleur_seuil(Y_val, proba_val_base)
pred_test_base = predire_avec_seuil(baseline.predict_proba(X_test), seuil_base)
res_base = scores(Y_test, pred_test_base)

with mlflow.start_run(run_name="baseline_tfidf_logreg"):
    mlflow.log_params({**params_base, "seuil": round(float(seuil_base), 2)})
    mlflow.log_metrics({f"test_{k}": v for k, v in res_base.items()})

print("Seuil choisi sur la validation :", round(float(seuil_base), 2))
print({k: round(v, 3) for k, v in res_base.items()})
print(classification_report(Y_test, pred_test_base, target_names=LABELS, digits=3, zero_division=0))

## 7. Modèle principal : CamemBERT fine-tuné (PyTorch)

CamemBERT est un modèle de langue pré-entraîné sur du texte français. On ajoute une couche de classification avec **7 sorties**,
chacune passée dans une sigmoïde (une probabilité indépendante par famille), et on entraîne tout le réseau avec une
perte d'**entropie croisée binaire** (`problem_type="multi_label_classification"`).

Durée sur un GPU T4 de Colab : environ 10 à 15 minutes.

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

set_seed(SEED)
MODEL_NAME = "almanach/camembert-base"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# vérifie que 128 tokens suffisent pour nos textes
longueurs = [len(tokenizer(t)["input_ids"]) for t in train_df["texte"].sample(1000, random_state=SEED)]
print("tokens : médiane", int(np.median(longueurs)), "| 95e percentile", int(np.percentile(longueurs, 95)))

def vers_dataset(d):
    ds = Dataset.from_dict({"texte": d["texte"].tolist(), "labels": d[LABELS].values.astype("float32").tolist()})
    return ds.map(lambda b: tokenizer(b["texte"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["texte"])

ds_train, ds_val, ds_test = vers_dataset(train_df), vers_dataset(val_df), vers_dataset(test_df)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type="multi_label_classification",
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id={l: i for i, l in enumerate(LABELS)},
)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = predire_avec_seuil(sigmoid(logits), 0.5)
    return scores(labels.astype(int), preds)

HP = {"learning_rate": 3e-5, "epochs": 4, "batch_size": 32, "weight_decay": 0.01, "warmup_ratio": 0.1, "max_len": MAX_LEN}

# nombre total de pas d'entraînement, et 10 % de "warmup" (montée progressive du learning rate)
import math
pas_total = math.ceil(len(ds_train) / HP["batch_size"]) * HP["epochs"]
warmup = int(HP["warmup_ratio"] * pas_total)

args = TrainingArguments(
    output_dir="checkpoints",
    learning_rate=HP["learning_rate"],
    num_train_epochs=HP["epochs"],
    per_device_train_batch_size=HP["batch_size"],
    per_device_eval_batch_size=64,
    weight_decay=HP["weight_decay"],
    warmup_steps=warmup,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,         # garde l'époque avec le meilleur F1 macro en validation
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",                    # on loggue nous-mêmes dans MLflow ci-dessous
    seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

## 8. Seuil, évaluation sur le test et enregistrement dans MLflow

In [ ]:
proba_val_cb = sigmoid(trainer.predict(ds_val).predictions)
seuil_cb = meilleur_seuil(Y_val, proba_val_cb)

proba_test_cb = sigmoid(trainer.predict(ds_test).predictions)
pred_test_cb = predire_avec_seuil(proba_test_cb, seuil_cb)
res_cb = scores(Y_test, pred_test_cb)

with mlflow.start_run(run_name="camembert_finetune"):
    mlflow.log_params({**HP, "model": MODEL_NAME, "seuil": round(float(seuil_cb), 2)})
    for entree in trainer.state.log_history:             # courbes d'entraînement et de validation par époque
        etape = int(entree.get("step", 0))
        for k, v in entree.items():
            if isinstance(v, (int, float)) and k not in ("step", "epoch"):
                mlflow.log_metric(k, v, step=etape)
    mlflow.log_metrics({f"test_{k}": v for k, v in res_cb.items()})

print("Seuil choisi sur la validation :", round(float(seuil_cb), 2))
print({k: round(v, 3) for k, v in res_cb.items()})
print(classification_report(Y_test, pred_test_cb, target_names=LABELS, digits=3, zero_division=0))

## 9. Comparaison baseline / CamemBERT (test 2026)

In [ ]:
comparaison = pd.DataFrame({
    "baseline TF-IDF": {**res_base, **dict(zip(LABELS, f1_score(Y_test, pred_test_base, average=None, zero_division=0)))},
    "CamemBERT": {**res_cb, **dict(zip(LABELS, f1_score(Y_test, pred_test_cb, average=None, zero_division=0)))},
}).round(3)
comparaison["écart"] = (comparaison["CamemBERT"] - comparaison["baseline TF-IDF"]).round(3)
comparaison

## 10. Analyse d'erreurs

On regarde les fiches où CamemBERT se trompe : quelles familles il confond, et si l'erreur vient du modèle ou d'une étiquette ambiguë.

In [ ]:
def noms(vecteur):
    return [LABELS[i] for i, v in enumerate(vecteur) if v == 1] or ["(aucune → autre)"]

erreurs = test_df.assign(
    vrai=[noms(v) for v in Y_test],
    predit=[noms(v) for v in pred_test_cb],
)
erreurs = erreurs[(Y_test != pred_test_cb).any(axis=1)]
print(f"{len(erreurs)} fiches mal classées sur {len(test_df)} ({len(erreurs)/len(test_df):.1%})")

# confusions les plus fréquentes
paires = erreurs.apply(lambda r: f"{'+'.join(r['vrai'])}  →  {'+'.join(r['predit'])}", axis=1)
print(paires.value_counts().head(10))

pd.set_option("display.max_colwidth", 200)
erreurs[["texte", "risques_encourus", "vrai", "predit"]].sample(min(15, len(erreurs)), random_state=SEED)

## 11. Export du modèle (pour l'API)

On sauvegarde le modèle, le tokenizer, la liste des familles et le seuil, puis on télécharge le tout, ainsi que le suivi MLflow.

In [ ]:
import shutil, os
OUT = "modele_rappelconso"
trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
with open(os.path.join(OUT, "config_api.json"), "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "seuil": round(float(seuil_cb), 2), "max_len": MAX_LEN}, f, ensure_ascii=False, indent=2)

shutil.make_archive(OUT, "zip", OUT)
os.makedirs("suivi_mlflow", exist_ok=True)
shutil.copy("mlflow.db", "suivi_mlflow/")
if os.path.isdir("mlruns"):
    shutil.copytree("mlruns", "suivi_mlflow/mlruns", dirs_exist_ok=True)
shutil.make_archive("suivi_mlflow", "zip", "suivi_mlflow")
files.download(OUT + ".zip")
files.download("suivi_mlflow.zip")

## 12. Test rapide : prédire une nouvelle fiche

In [ ]:
def predire(texte):
    enc = tokenizer(texte, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        probas = torch.sigmoid(trainer.model(**enc).logits)[0].cpu().numpy()
    familles = [(LABELS[i], round(float(p), 3)) for i, p in enumerate(probas) if p >= seuil_cb]
    return familles or [("autre / à vérifier", round(float(probas.max()), 3))]

exemple = "Produit : fromage au lait cru | Catégorie : alimentation / lait et produits laitiers | Motif : présence de listeria monocytogenes détectée lors d'un autocontrôle"
predire(exemple)